In [1]:
# 1

import numpy as np
import pandas as pd
import yfinance as yf
import os

rng = np.random.default_rng(0)
pd.set_option("display.width", 200)

TICKERS = {"VIX": "^VIX", "SPX": "^GSPC", "OVX": "^OVX", "GVZ": "^GVZ"}
SECTORS = ["XLB", "XLE", "XLF", "XLI", "XLK", "XLP", "XLU", "XLV", "XLY", "XLRE", "XLC"]
SEC9 = ["XLB", "XLE", "XLF", "XLI", "XLK", "XLP", "XLU", "XLV", "XLY"]

WIN, MINP = 60, 40
Z, F = 1.5, 1.0
HORIZONS = [1, 2, 3, 4, 5]
N_PERM = 5000
ROLL_W = 750

PERIODS = [("1990-1999", 1990, 1999), ("2000-2009", 2000, 2009),
           ("2010-2019", 2010, 2019), ("2020-2026", 2020, 2026),
           ("2023-2026", 2023, 2026)]


def welch(a, b):
    a, b = a.dropna(), b.dropna()
    na, nb = len(a), len(b)
    va, vb = a.var(ddof=1) / na, b.var(ddof=1) / nb
    t = (a.mean() - b.mean()) / np.sqrt(va + vb)
    return a.mean(), b.mean(), t, na, nb


def perm_diff(y, sig, n_iter=N_PERM):
    y = np.asarray(y, dtype=float)
    sig = np.asarray(sig, dtype=bool)
    ok = ~np.isnan(y)
    obs = np.nanmean(y[sig & ok]) - np.nanmean(y[~sig & ok])
    n = len(y)
    null = np.empty(n_iter)
    for i in range(n_iter):
        sh = np.roll(sig, rng.integers(1, n))
        null[i] = np.nanmean(y[sh & ok]) - np.nanmean(y[~sh & ok])
    mu, sd = null.mean(), null.std()
    return obs, sd, (obs - mu) / sd, (np.abs(null - mu) >= abs(obs - mu)).mean()


def compare(y, sig_mask, base_mask, scale=1.0, perm=True):
    y = pd.Series(y) if not isinstance(y, pd.Series) else y
    ma, mb, t, na, nb = welch(y[sig_mask], y[base_mask])
    out = {"n_sig": na, "n_base": nb, "signal": ma * scale,
           "base": mb * scale, "diff": (ma - mb) * scale, "t_welch": t}
    if perm:
        _, _, z, p = perm_diff(y.values, sig_mask)
        out["perm_z"], out["p_perm"] = z, p
    return out

KeyboardInterrupt: 

In [ ]:
# 2

os.makedirs("cache", exist_ok=True)
CACHE = "cache/prices_v2.parquet"

INDICES = ["SPX", "NDX", "DJI", "RUT"]
TICKERS = {"VIX": "^VIX", "SPX": "^GSPC", "NDX": "^NDX", "DJI": "^DJI", "RUT": "^RUT",
           "OVX": "^OVX", "GVZ": "^GVZ"}


def fetch(tickers, start="1990-01-01"):
    raw = yf.download(list(tickers), start=start, auto_adjust=True,
                      progress=False, group_by="column")
    return raw["Close"] if isinstance(raw.columns, pd.MultiIndex) else raw


if os.path.exists(CACHE):
    px_raw = pd.read_parquet(CACHE)
else:
    a = fetch(list(TICKERS.values())).rename(columns={v: k for k, v in TICKERS.items()})
    b = fetch(SECTORS)
    px_raw = a.join(b, how="outer").sort_index()
    px_raw.index = pd.to_datetime(px_raw.index).tz_localize(None)
    px_raw.to_parquet(CACHE)

px = px_raw[px_raw["SPX"].notna()].copy()

cov = pd.DataFrame({
    "start": px.apply(lambda s: s.first_valid_index()),
    "end": px.apply(lambda s: s.last_valid_index()),
    "n_obs": px.notna().sum(),
    "n_gap": px.apply(lambda s: s.loc[s.first_valid_index():s.last_valid_index()].isna().sum()
                      if s.first_valid_index() is not None else np.nan),
})
print("[coverage]")
print(cov.to_string())

print("\n[trading days per year]")
print(px.groupby(px.index.year).size().to_string())

print("\n[validation]")
print("rows (SPX calendar):", len(px), "|", px.index.min().date(), "~", px.index.max().date())
print("index monotonic:", px.index.is_monotonic_increasing, "| duplicates:", int(px.index.duplicated().sum()))

lr = np.log(px[["VIX"] + INDICES]).diff()
print("\n[corr with dlog VIX]")
print(lr.corr().loc["VIX", INDICES].round(3).to_string())
print("\n[corr among indices]")
print(lr[INDICES].corr().round(3).to_string())

[coverage]
            start        end  n_obs  n_gap
Ticker                                    
DJI    1992-01-02 2026-07-24   8701      0
SPX    1990-01-02 2026-07-24   9207      0
GVZ    2008-06-03 2026-07-24   4564      0
NDX    1990-01-02 2026-07-24   9207      0
OVX    2007-05-10 2026-07-24   4832      0
RUT    1990-01-02 2026-07-24   9207      0
VIX    1990-01-02 2026-07-24   9207      0
XLB    1998-12-22 2026-07-24   6938      0
XLC    2018-06-19 2026-07-24   2035      0
XLE    1998-12-22 2026-07-24   6938      0
XLF    1998-12-22 2026-07-24   6938      0
XLI    1998-12-22 2026-07-24   6938      0
XLK    1998-12-22 2026-07-24   6938      0
XLP    1998-12-22 2026-07-24   6938      0
XLRE   2015-10-08 2026-07-24   2713      0
XLU    1998-12-22 2026-07-24   6938      0
XLV    1998-12-22 2026-07-24   6938      0
XLY    1998-12-22 2026-07-24   6938      0

[trading days per year]
Date
1990    253
1991    253
1992    254
1993    253
1994    252
1995    252
1996    254
1997    253
199

In [ ]:
# 3

d = pd.DataFrame(index=px.index)
for ix in INDICES:
    d[f"{ix.lower()}_ret"] = np.log(px[ix]).diff()
    sd = d[f"{ix.lower()}_ret"].rolling(WIN, min_periods=MINP).std().shift(1)
    d[f"{ix.lower()}_z"] = d[f"{ix.lower()}_ret"] / sd
d["spx_ret"] = d["spx_ret"]
d["spx_z"] = d["spx_z"]

for v in ["VIX", "OVX", "GVZ"]:
    ch = np.log(px[v]).diff()
    d[f"{v.lower()}_ch"] = ch
    d[f"{v.lower()}_z"] = ch / ch.rolling(WIN, min_periods=MINP).std().shift(1)

base = d.dropna(subset=["vix_z", "spx_z"]).copy()
hi = base["vix_z"] > Z
fl = base["spx_z"].abs() < F

grp = pd.Series(index=base.index, dtype=object)
grp[hi & fl] = "hi_flat"
grp[hi & ~fl] = "hi_move"
grp[~hi & fl] = "lo_flat"
grp[~hi & ~fl] = "lo_move"

SIG = (grp == "hi_flat").values
BASE = (grp == "lo_flat").values

MASKS = {}
for ix in INDICES:
    zi = base[f"{ix.lower()}_z"]
    ok = zi.notna().values
    MASKS[ix] = {"sig": (hi & (zi.abs() < F)).values & ok,
                 "base": (~hi & (zi.abs() < F)).values & ok}

n = len(base)
print("[sample]", base.index.min().date(), "~", base.index.max().date(), "| n =", n)
print("\n[2x2 groups, SPX-based]")
print(grp.value_counts().to_string())

print("\n[signal definition per index: VIX spike + that index quiet]")
rows = []
for ix in INDICES:
    s = MASKS[ix]["sig"]
    b = MASKS[ix]["base"]
    ov = (s & SIG).sum()
    rows.append({"index": ix, "n_obs": int((s | b).sum()), "n_signal": int(s.sum()),
                 "n_base": int(b.sum()), "per_year": round(s.sum() / (len(base) / 252), 1),
                 "overlap_with_SPX_sig": int(ov),
                 "overlap_pct": round(100 * ov / max(s.sum(), 1), 1),
                 "monday_share": round((base.index[s].dayofweek == 0).mean(), 3)})
print(pd.DataFrame(rows).to_string(index=False))

print("\n[observed / expected under independence, per index]")
rows = []
for ix in INDICES:
    zi = base[f"{ix.lower()}_z"]
    ok = zi.notna()
    ps = (hi & ok).sum() / ok.sum()
    pf = ((zi.abs() < F) & ok).sum() / ok.sum()
    k = MASKS[ix]["sig"].sum()
    rows.append({"index": ix, "P(spike)": round(ps, 4), "P(quiet)": round(pf, 4),
                 "expected": round(ps * pf * ok.sum(), 1), "observed": int(k),
                 "ratio": round(k / (ps * pf * ok.sum()), 3)})
print(pd.DataFrame(rows).to_string(index=False))

sig_idx = base.index[SIG]
mag = pd.DataFrame({"vix_pct": (np.exp(d["vix_ch"]) - 1) * 100, "vix_z": d["vix_z"],
                    "decade": px.index.year // 10 * 10}).loc[sig_idx]
print("\n[SPX signal: median VIX move by decade]")
print(mag.groupby("decade")[["vix_pct", "vix_z"]].median().round(2).to_string())
print("VIX move < 5%:", int((mag["vix_pct"] < 5).sum()), "/", len(mag))

[sample] 1990-03-01 ~ 2026-07-24 | n = 9166

[2x2 groups, SPX-based]
lo_flat    6373
lo_move    2154
hi_move     502
hi_flat     137

[signal definition per index: VIX spike + that index quiet]
index  n_obs  n_signal  n_base  per_year  overlap_with_SPX_sig  overlap_pct  monday_share
  SPX   6510       137    6373       3.8                   137        100.0         0.453
  NDX   6374       192    6182       5.3                   108         56.2         0.375
  DJI   6136       141    5995       3.9                   109         77.3         0.433
  RUT   6384       173    6211       4.8                    94         54.3         0.329

[observed / expected under independence, per index]
index  P(spike)  P(quiet)  expected  observed  ratio
  SPX    0.0697    0.7102     453.8       137  0.302
  NDX    0.0697    0.6954     444.4       192  0.432
  DJI    0.0703    0.7085     431.5       141  0.327
  RUT    0.0697    0.6965     445.1       173  0.389

[SPX signal: median VIX move by decad

In [ ]:
# 4

END = base.index.max()
W2M = base.index >= (END - pd.DateOffset(months=2))
print("[recent window]", base.index[W2M].min().date(), "~", END.date(),
      "| trading days:", int(W2M.sum()))

zs = [0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 2.0]
fs = [0.75, 1.0, 1.5, 2.0]

print("\n[signal count in recent 2 months]")
cnt2 = pd.DataFrame(index=[f"z>{z}" for z in zs], columns=[f"|s|<{f}" for f in fs], dtype=int)
for z in zs:
    for f in fs:
        cnt2.loc[f"z>{z}", f"|s|<{f}"] = int(((base["vix_z"] > z) & (base["spx_z"].abs() < f) & W2M).sum())
print(cnt2.to_string())

print("\n[implied full-sample frequency, signals per year]")
freq = pd.DataFrame(index=[f"z>{z}" for z in zs], columns=[f"|s|<{f}" for f in fs], dtype=float)
yrs_total = len(base) / 252
for z in zs:
    for f in fs:
        k = ((base["vix_z"] > z) & (base["spx_z"].abs() < f)).sum()
        freq.loc[f"z>{z}", f"|s|<{f}"] = round(k / yrs_total, 1)
print(freq.to_string())

print("\n[what each z threshold means: percentile of daily VIX moves, and median move]")
rows = []
for z in zs:
    m = base["vix_z"] > z
    pct = 100 * (1 - m.mean())
    mv = d["vix_ch"].reindex(base.index)[m]
    rows.append({"z": z, "pctile": round(pct, 1), "n_days": int(m.sum()),
                 "median_pct_move": round((np.exp(mv.median()) - 1) * 100, 2),
                 "min_pct_move": round((np.exp(mv.min()) - 1) * 100, 2)})
print(pd.DataFrame(rows).to_string(index=False))

print("\n[recent 2 months, day by day: all days with any VIX increase]")
rec = pd.DataFrame({
    "VIX": px["VIX"].reindex(base.index),
    "vix_pct": (np.exp(d["vix_ch"]) - 1).reindex(base.index) * 100,
    "vix_z": base["vix_z"],
    "spx_pct": (np.exp(d["spx_ret"]) - 1).reindex(base.index) * 100,
    "spx_z": base["spx_z"],
})[W2M]
rec = rec[rec["vix_pct"] > 0].sort_values("vix_z", ascending=False)
print(rec.round(2).to_string())

[recent window] 2026-05-26 ~ 2026-07-24 | trading days: 42

[signal count in recent 2 months]
        |s|<0.75  |s|<1.0  |s|<1.5  |s|<2.0
z>0.25       6.0      8.0     11.0     13.0
z>0.5        5.0      6.0      9.0     11.0
z>0.75       1.0      2.0      5.0      7.0
z>1.0        0.0      1.0      4.0      6.0
z>1.25       0.0      1.0      4.0      6.0
z>1.5        0.0      1.0      4.0      5.0
z>2.0        0.0      0.0      0.0      0.0

[implied full-sample frequency, signals per year]
        |s|<0.75  |s|<1.0  |s|<1.5  |s|<2.0
z>0.25      43.9     54.8     70.7     79.3
z>0.5       27.0     35.1     49.0     57.1
z>0.75      14.7     20.5     31.4     38.7
z>1.0        8.2     11.9     19.8     26.0
z>1.25       4.0      6.4     11.7     16.9
z>1.5        2.2      3.8      7.1     10.9
z>2.0        0.4      0.9      2.1      4.2

[what each z threshold means: percentile of daily VIX moves, and median move]
   z  pctile  n_days  median_pct_move  min_pct_move
0.25    64.9    3213

In [ ]:
# 5 
fwd = {}
for ix in INDICES:
    f = pd.DataFrame(index=base.index)
    for h in HORIZONS:
        f[f"h{h}"] = d[f"{ix.lower()}_ret"].shift(-h).reindex(base.index)
    f["cum1_5"] = f[[f"h{h}" for h in HORIZONS]].sum(axis=1, min_count=len(HORIZONS))
    fwd[ix] = f

print("[A. VIX spike + that index quiet -> that index forward returns]")
rows = []
for ix in INDICES:
    s, b = MASKS[ix]["sig"], MASKS[ix]["base"]
    for c in ["h1", "h2", "cum1_5"]:
        ma, mb, t, na, nb = welch(fwd[ix][c][s], fwd[ix][c][b])
        _, _, pz, pp = perm_diff(fwd[ix][c].values, s)
        rows.append({"index": ix, "horizon": c, "n_sig": na,
                     "signal_bps": round(ma * 1e4, 1), "base_bps": round(mb * 1e4, 1),
                     "diff_bps": round((ma - mb) * 1e4, 1), "t_welch": round(t, 2),
                     "perm_z": round(pz, 2), "p_perm": round(pp, 4)})
print(pd.DataFrame(rows).to_string(index=False))

print("\n[B. all horizons]")
tab = pd.DataFrame(index=[f"h{h}" for h in HORIZONS] + ["cum1_5"], columns=INDICES, dtype=float)
pv = tab.copy()
for ix in INDICES:
    s, b = MASKS[ix]["sig"], MASKS[ix]["base"]
    for c in tab.index:
        ma, mb, t, na, nb = welch(fwd[ix][c][s], fwd[ix][c][b])
        _, _, pz, pp = perm_diff(fwd[ix][c].values, s)
        tab.loc[c, ix] = round((ma - mb) * 1e4, 1)
        pv.loc[c, ix] = round(pp, 3)
print("diff (bps):")
print(tab.to_string())
print("\np_perm:")
print(pv.to_string())

print("\n[C. distribution of h1]")
rows = []
for ix in INDICES:
    s, b = MASKS[ix]["sig"], MASKS[ix]["base"]
    for lbl, m in [("signal", s), ("base", b)]:
        x = fwd[ix]["h1"][m].dropna()
        rows.append({"index": ix, "group": lbl, "n": len(x),
                     "pct_neg": round((x < 0).mean() * 100, 1),
                     "q05": round(x.quantile(0.05) * 1e4, 1),
                     "median": round(x.median() * 1e4, 1),
                     "q95": round(x.quantile(0.95) * 1e4, 1)})
print(pd.DataFrame(rows).to_string(index=False))

[A. VIX spike + that index quiet -> that index forward returns]
index horizon  n_sig  signal_bps  base_bps  diff_bps  t_welch  perm_z  p_perm
  SPX      h1    137        11.7       2.4       9.3     1.23    0.88  0.3778
  SPX      h2    137        17.1       2.5      14.6     1.75    1.42  0.1470
  SPX  cum1_5    137        33.8      15.0      18.9     1.07    0.85  0.3846
  NDX      h1    192         4.7       4.6       0.1     0.01   -0.08  0.9452
  NDX      h2    192        33.8       4.8      29.0     2.13    2.34  0.0176
  NDX  cum1_5    192        44.9      23.5      21.4     0.83    0.70  0.4796
  DJI      h1    141        10.3       2.8       7.6     1.01    0.78  0.4234
  DJI      h2    141        10.9       2.5       8.5     1.07    0.81  0.4078
  DJI  cum1_5    141        44.3      14.5      29.8     1.91    1.45  0.1506
  RUT      h1    173        16.8       1.5      15.3     1.69    1.27  0.2060
  RUT      h2    172        20.1       3.1      17.0     1.90    1.63  0.1052


In [ ]:
# 6
FS = fwd["SPX"]

sec_ret = np.log(px[SECTORS]).diff()
r9 = np.log(px[SEC9]).diff()

rot_raw = pd.DataFrame(index=px.index)
rot_raw["disp"] = np.log(sec_ret.std(axis=1, ddof=1))
rot_raw["rankcorr"] = sec_ret.corrwith(sec_ret.shift(1), axis=1, method="spearman")
rot_raw["n_sec"] = sec_ret.notna().sum(axis=1)

ctrl = pd.DataFrame({"y": rot_raw["disp"], "absz": d["spx_z"].abs(),
                     "n9": (rot_raw["n_sec"] == 9).astype(float),
                     "n10": (rot_raw["n_sec"] == 10).astype(float)}).dropna()
Xc = np.column_stack([np.ones(len(ctrl)), ctrl["absz"], ctrl["n9"], ctrl["n10"]])
yc = ctrl["y"].values
resid = np.full(len(ctrl), np.nan)
for i in range(ROLL_W, len(ctrl)):
    c, *_ = np.linalg.lstsq(Xc[i - ROLL_W:i], yc[i - ROLL_W:i], rcond=None)
    resid[i] = yc[i] - Xc[i] @ c
rot_raw["disp_resid"] = pd.Series(resid, index=ctrl.index)

y_rank = rot_raw["rankcorr"].shift(-1).reindex(base.index)
y_disp = rot_raw["disp_resid"].shift(-1).reindex(base.index)

print("[control regression: log(disp) ~ |spx_z| + n_sec dummies, rolling 750d]")
cf, *_ = np.linalg.lstsq(Xc, yc, rcond=None)
print("full-sample coef [const, |z|, n9, n10]:", np.round(cf, 4),
      "| R2:", round(1 - np.var(yc - Xc @ cf) / np.var(yc), 4))

print("\n[A. h1: hi_flat vs lo_flat]")
rows = []
for m in ["disp", "disp_resid", "rankcorr"]:
    y1 = rot_raw[m].shift(-1).reindex(base.index)
    r = compare(y1, SIG, BASE)
    rows.append({"measure": m, **{k: (round(v, 4) if isinstance(v, float) else v) for k, v in r.items()}})
print(pd.DataFrame(rows).to_string(index=False))

print("\n[B. disp_resid across horizons]")
rows = []
for h in [0] + HORIZONS:
    y = rot_raw["disp_resid"].shift(-h).reindex(base.index)
    r = compare(y, SIG, BASE)
    rows.append({"h": h, **{k: (round(v, 4) if isinstance(v, float) else v) for k, v in r.items()}})
print(pd.DataFrame(rows).to_string(index=False))

print("\n[C. persistence control: is h1 just carryover from h0?]")
rr0 = rot_raw["disp_resid"].reindex(base.index)
rr1 = rot_raw["disp_resid"].shift(-1).reindex(base.index)
ar = pd.DataFrame({"y": rr1, "x": rr0}).dropna()
sl, ic = np.polyfit(ar["x"].values, ar["y"].values, 1)
print(f"AR(1) slope: {sl:.4f} | corr(h0,h1): {ar['x'].corr(ar['y']):.3f}")
rows = []
for lbl, y in [("h1 raw", rr1), ("h1 | h0", rr1 - (sl * rr0 + ic)), ("h1 - h0", rr1 - rr0)]:
    r = compare(y, SIG, BASE)
    rows.append({"spec": lbl, **{k: (round(v, 4) if isinstance(v, float) else v) for k, v in r.items()}})
print(pd.DataFrame(rows).to_string(index=False))

[control regression: log(disp) ~ |spx_z| + n_sec dummies, rolling 750d]
full-sample coef [const, |z|, n9, n10]: [-5.0381  0.1708 -0.0778 -0.2786] | R2: 0.0779

[A. h1: hi_flat vs lo_flat]
   measure  n_sig  n_base  signal    base    diff  t_welch  perm_z  p_perm
      disp     92    4843 -5.1464 -4.9955 -0.1509  -2.7217 -3.1424  0.0022
disp_resid     79    4323 -0.2474 -0.0672 -0.1803  -3.2907 -4.1258  0.0000
  rankcorr     92    4842 -0.0397 -0.0275 -0.0122  -0.2784 -0.2937  0.7722

[B. disp_resid across horizons]
 h  n_sig  n_base  signal    base    diff  t_welch  perm_z  p_perm
 0     79    4323 -0.1227 -0.0555 -0.0672  -1.2582 -1.4443  0.1442
 1     79    4323 -0.2474 -0.0672 -0.1803  -3.2907 -4.1263  0.0000
 2     79    4324 -0.1289 -0.0649 -0.0640  -1.3209 -1.5716  0.1150
 3     79    4324 -0.1467 -0.0609 -0.0858  -1.7270 -1.9281  0.0498
 4     79    4324 -0.0987 -0.0662 -0.0325  -0.6837 -0.9438  0.3524
 5     79    4324 -0.1619 -0.0624 -0.0995  -2.0138 -2.2872  0.0228

[C. persi

In [ ]:
# 7

VOLS = [("vix", "VIX"), ("ovx", "OVX"), ("gvz", "GVZ")]
w08 = base.index.year >= 2008


def vol_masks(v, window=None):
    zz = d[f"{v}_z"].reindex(base.index)
    ok = zz.notna().values
    if window is not None:
        ok = ok & window
    return ((zz > Z) & fl).values & ok, ((zz <= Z) & fl).values & ok


print("[primary: each vol index on its own full history]")
rows = []
for v, V in VOLS:
    s, b = vol_masks(v)
    for lbl, y in [("rankcorr", y_rank), ("disp_resid", y_disp)]:
        ma, mb, t, na, nb = welch(y[s], y[b])
        _, sd, pz, pp = perm_diff(y.values, s)
        rows.append({"vol": V, "since": base.index[s].min().year, "measure": lbl,
                     "n_sig": na, "n_base": nb, "signal": round(ma, 4), "base": round(mb, 4),
                     "diff": round(ma - mb, 4), "t_welch": round(t, 2),
                     "perm_z": round(pz, 2), "p_perm": round(pp, 4)})
print(pd.DataFrame(rows).to_string(index=False))

print("\n[forward SPX returns, bps]")
rows = []
for v, V in VOLS:
    s, b = vol_masks(v)
    for c in ["h1", "h2"]:
        ma, mb, t, na, nb = welch(FS[c][s], FS[c][b])
        _, _, pz, pp = perm_diff(FS[c].values, s)
        rows.append({"vol": V, "horizon": c, "n_sig": na,
                     "signal": round(ma * 1e4, 1), "base": round(mb * 1e4, 1),
                     "diff": round((ma - mb) * 1e4, 1), "t_welch": round(t, 2),
                     "perm_z": round(pz, 2), "p_perm": round(pp, 4)})
print(pd.DataFrame(rows).to_string(index=False))

print("\n[diagnostic: why the 2008+ common sample was discarded]")
print("disp_resid, VIX only")
rows = []
for lbl, win in [("full (1990+)", None), ("common (2008+)", w08)]:
    s, b = vol_masks("vix", win)
    yy = y_disp.copy()
    if win is not None:
        yy[~win] = np.nan
    ma, mb, t, na, nb = welch(yy[s], yy[b])
    _, sd, pz, pp = perm_diff(yy.values, s)
    rows.append({"sample": lbl, "n_sig": na, "diff": round(ma - mb, 4),
                 "se_welch": round((ma - mb) / t, 4), "perm_sd": round(sd, 4),
                 "t_welch": round(t, 2), "perm_z": round(pz, 2), "p_perm": round(pp, 4)})
dg = pd.DataFrame(rows)
print(dg.to_string(index=False))
print(f"\nperm_sd ratio (common / full): {dg['perm_sd'].iloc[1] / dg['perm_sd'].iloc[0]:.1f}x"
      f"  vs  se_welch ratio: {dg['se_welch'].iloc[1] / dg['se_welch'].iloc[0]:.1f}x")

[primary: each vol index on its own full history]
vol  since    measure  n_sig  n_base  signal    base    diff  t_welch  perm_z  p_perm
VIX   1990   rankcorr     92    4842 -0.0397 -0.0275 -0.0122    -0.28   -0.32  0.7516
VIX   1990 disp_resid     79    4323 -0.2474 -0.0672 -0.1803    -3.29   -4.10  0.0000
OVX   2007   rankcorr    179    3251 -0.0420 -0.0238 -0.0182    -0.56   -0.41  0.6678
OVX   2007 disp_resid    179    3251  0.0120 -0.0271  0.0391     1.05    0.90  0.3734
GVZ   2008   rankcorr    192    3060 -0.0340 -0.0191 -0.0149    -0.49   -0.19  0.8454
GVZ   2008 disp_resid    192    3060 -0.0312 -0.0487  0.0175     0.48    0.39  0.7074

[forward SPX returns, bps]
vol horizon  n_sig  signal  base  diff  t_welch  perm_z  p_perm
VIX      h1    137    11.7   2.4   9.3     1.23    0.86  0.3918
VIX      h2    137    17.1   2.5  14.6     1.75    1.43  0.1494
OVX      h1    179     2.0   3.5  -1.5    -0.13   -0.17  0.8594
OVX      h2    179    17.5   2.8  14.6     1.43    1.66  0.0954


In [ ]:
# 8

lr_spx = np.log(px["SPX"]).diff()
lr_vix = np.log(px["VIX"]).diff()

print("[index and alignment]")
print("monotonic:", px.index.is_monotonic_increasing,
      "| duplicates:", int(px.index.duplicated().sum()),
      "| d aligned to px:", d.index.equals(px.index),
      "| base subset of d:", base.index.isin(d.index).all())

errs_s, errs_v = [], []
for ti in rng.choice(np.arange(WIN + 5, len(px)), 100, replace=False):
    hs, hv = lr_spx.iloc[ti - WIN:ti], lr_vix.iloc[ti - WIN:ti]
    if hs.notna().sum() >= MINP:
        errs_s.append(abs(lr_spx.iloc[ti] / hs.std(ddof=1) - d["spx_z"].iloc[ti]))
    if hv.notna().sum() >= MINP:
        errs_v.append(abs(lr_vix.iloc[ti] / hv.std(ddof=1) - d["vix_z"].iloc[ti]))
print("\n[look-ahead: manual recompute of z-scores, 100 random dates]")
print("max abs err spx_z:", np.nanmax(errs_s), "| vix_z:", np.nanmax(errs_v))

pos = d.index.get_indexer(base.index)
bad = 0
for ix in INDICES:
    for h in HORIZONS:
        tgt = np.where(pos + h < len(d), d[f"{ix.lower()}_ret"].values[np.clip(pos + h, 0, len(d) - 1)], np.nan)
        bad += int((np.abs(np.nan_to_num(tgt) - np.nan_to_num(fwd[ix][f"h{h}"].values)) > 1e-12).sum())
print("\n[forward alignment, all indices]")
print("misaligned cells:", bad)
print("cum1_5 == sum(h1..h5):",
      all(bool(np.nanmax(np.abs(fwd[ix][[f"h{h}" for h in HORIZONS]].sum(axis=1, min_count=5)
                                - fwd[ix]["cum1_5"])) < 1e-12) for ix in INDICES))

chk = []
for k in rng.choice(np.arange(ROLL_W + 10, len(ctrl)), 5, replace=False):
    dt = ctrl.index[k]
    c, *_ = np.linalg.lstsq(Xc[k - ROLL_W:k], yc[k - ROLL_W:k], rcond=None)
    chk.append(abs((yc[k] - Xc[k] @ c) - rot_raw.loc[dt, "disp_resid"]))
print("\n[rolling residual uses only past window]")
print("max abs err:", max(chk))
print("first valid disp_resid:", rot_raw["disp_resid"].first_valid_index().date(),
      "| ctrl start:", ctrl.index.min().date())

print("\n[degenerate observations]")
print("zero dlog VIX days:", int((lr_vix == 0).sum()),
      "| among signals:", int((lr_vix.reindex(base.index[SIG]) == 0).sum()))
print("zero dlog SPX days:", int((lr_spx == 0).sum()))
print("signal days with any sector NaN:",
      int(sec_ret.reindex(base.index[SIG]).isna().any(axis=1).sum()), "/", int(SIG.sum()))
print("sector count on signal days:")
print(rot_raw["n_sec"].reindex(base.index[SIG]).value_counts().sort_index().to_string())

print("\n[signal clustering]")
s_ser = pd.Series(SIG.astype(int), index=base.index)
gaps = np.diff(np.where(SIG)[0])
print("autocorr lag1-5:", [round(s_ser.autocorr(l), 3) for l in range(1, 6)])
print("gap median:", int(np.median(gaps)), "| gaps <= 5d:", int((gaps <= 5).sum()), "/", len(gaps),
      "| max consecutive:", int(s_ser.groupby((s_ser != s_ser.shift()).cumsum()).cumsum().max()))

[index and alignment]
monotonic: True | duplicates: 0 | d aligned to px: True | base subset of d: True

[look-ahead: manual recompute of z-scores, 100 random dates]
max abs err spx_z: 3.5083047578154947e-14 | vix_z: 6.661338147750939e-15

[forward alignment, all indices]
misaligned cells: 0
cum1_5 == sum(h1..h5): True

[rolling residual uses only past window]
max abs err: 0.0
first valid disp_resid: 2001-12-19 | ctrl start: 1998-12-23

[degenerate observations]
zero dlog VIX days: 51 | among signals: 0
zero dlog SPX days: 5
signal days with any sector NaN: 109 / 137
sector count on signal days:
n_sec
0     45
9     57
10     7
11    28

[signal clustering]
autocorr lag1-5: [np.float64(-0.008), np.float64(-0.0), np.float64(0.022), np.float64(0.014), np.float64(0.029)]
gap median: 54 | gaps <= 5d: 18 / 136 | max consecutive: 2


In [ ]:
# 9

print("[A. threshold surface: hi_flat vs lo_flat at each (Z, F)]")
rows = []
for zt in [1.25, 1.5, 1.75, 2.0]:
    for ft in [0.5, 0.75, 1.0, 1.25]:
        s = ((base["vix_z"] > zt) & (base["spx_z"].abs() < ft)).values
        b = ((base["vix_z"] <= zt) & (base["spx_z"].abs() < ft)).values
        if s.sum() < 10:
            continue
        mr, br, tr, nr, _ = welch(FS["h1"][s], FS["h1"][b])
        md, bd, td, nd, _ = welch(y_disp[s], y_disp[b])
        _, _, pzd, ppd = perm_diff(y_disp.values, s)
        rows.append({"Z": zt, "F": ft, "n_ret": nr, "h1_diff_bps": round((mr - br) * 1e4, 1),
                     "t_ret": round(tr, 2), "n_disp": nd, "disp_diff": round(md - bd, 4),
                     "t_disp": round(td, 2), "perm_z": round(pzd, 2), "p_perm": round(ppd, 4)})
print(pd.DataFrame(rows).to_string(index=False))

print("\n[B. day-of-week: signal is 45% Monday]")
dm = pd.Series({k: rot_raw["disp_resid"].reindex(base.index)[base.index.dayofweek == i].mean()
                for i, k in enumerate(["Mon", "Tue", "Wed", "Thu", "Fri"])})
print("mean disp_resid by weekday:", dm.round(4).to_dict())

nm = base.index.dayofweek != 0
rows = []
for lbl, m in [("all days", np.ones(len(base), dtype=bool)), ("Mon excluded", nm)]:
    yy = y_disp.copy()
    yy[~m] = np.nan
    r = compare(yy, SIG & m, BASE & m)
    rows.append({"spec": lbl, **{k: (round(v, 4) if isinstance(v, float) else v) for k, v in r.items()}})

lv_dm = lr_vix - lr_vix.groupby(lr_vix.index.dayofweek).transform("mean")
z_dm = (lv_dm / lv_dm.rolling(WIN, min_periods=MINP).std().shift(1)).reindex(base.index)
sig_dm = ((z_dm > Z) & fl).values
r = compare(y_disp, sig_dm, BASE)
rows.append({"spec": "DOW-demeaned VIX", **{k: (round(v, 4) if isinstance(v, float) else v) for k, v in r.items()}})
print()
print(pd.DataFrame(rows).to_string(index=False))
print("Monday share after demeaning:", round((base.index[sig_dm].dayofweek == 0).mean(), 3),
      "| original:", round((base.index[SIG].dayofweek == 0).mean(), 3))

print("\n[C. z-matched: flat vs move within narrow vix_z bands]")
rows = []
for lo, hiz in [(1.5, 1.8), (1.8, 2.1), (2.1, 2.6), (2.6, 99)]:
    inb = (base["vix_z"] > lo) & (base["vix_z"] <= hiz)
    a, b = (inb & fl).values, (inb & ~fl).values
    md, bd, td, nd, nb = welch(y_disp[a], y_disp[b])
    if nd < 5 or nb < 5:
        continue
    rows.append({"band": f"{lo}-{hiz}", "n_flat": nd, "n_move": nb,
                 "medz_flat": round(base.loc[a, "vix_z"].median(), 2),
                 "medz_move": round(base.loc[b, "vix_z"].median(), 2),
                 "flat": round(md, 4), "move": round(bd, 4),
                 "diff": round(md - bd, 4), "t_welch": round(td, 2)})
print(pd.DataFrame(rows).to_string(index=False))

[A. threshold surface: hi_flat vs lo_flat at each (Z, F)]
   Z    F  n_ret  h1_diff_bps  t_ret  n_disp  disp_diff  t_disp  perm_z  p_perm
1.25 0.50     86         -0.7  -0.08      31    -0.3556   -4.05   -4.84  0.0000
1.25 0.75    147         -1.6  -0.25      72    -0.2041   -3.32   -4.21  0.0000
1.25 1.00    234          1.8   0.31     135    -0.1209   -2.81   -3.53  0.0000
1.25 1.25    332         -1.2  -0.23     196    -0.0901   -2.63   -3.15  0.0024
1.50 0.50     45          1.1   0.10      15    -0.3597   -3.06   -3.92  0.0006
1.50 0.75     79         10.0   1.21      37    -0.3186   -3.91   -4.91  0.0000
1.50 1.00    137          9.3   1.23      79    -0.1803   -3.29   -4.12  0.0000
1.50 1.25    201          3.1   0.44     118    -0.1276   -3.00   -3.48  0.0014
1.75 0.50     24         13.2   0.85      11    -0.3618   -2.44   -3.14  0.0006
1.75 0.75     39         14.5   1.22      21    -0.3679   -3.75   -4.11  0.0000
1.75 1.00     72         11.1   1.27      44    -0.2436   -3.8

In [ ]:
# 10

alt = pd.DataFrame(index=px.index)
alt["disp_resid"] = rot_raw["disp_resid"]
alt["sd9"] = np.log(r9.std(axis=1, ddof=1))
alt["mad9"] = np.log(r9.sub(r9.mean(axis=1), axis=0).abs().mean(axis=1))
bt = sec_ret.rolling(250, min_periods=150).cov(d["spx_ret"]).div(
    d["spx_ret"].rolling(250, min_periods=150).var(), axis=0).shift(1)
alt["beta_disp"] = np.log((sec_ret - bt.mul(d["spx_ret"], axis=0)).std(axis=1, ddof=1))
alt["rankcorr"] = rot_raw["rankcorr"]

print("[A. alternative rotation measures, h1]")
rows = []
for c in alt.columns:
    y1 = alt[c].shift(-1).reindex(base.index)
    r = compare(y1, SIG, BASE)
    rows.append({"measure": c, **{k: (round(v, 4) if isinstance(v, float) else v) for k, v in r.items()}})
print(pd.DataFrame(rows).to_string(index=False))

print("\n[B. outlier robustness: disp_resid h1]")
a, b = y_disp[SIG].dropna(), y_disp[BASE].dropna()
lo_, hi_ = y_disp.quantile([0.01, 0.99])
print(f"mean diff      : {a.mean() - b.mean():.4f}")
print(f"median diff    : {a.median() - b.median():.4f}")
print(f"winsorised 1/99: {a.clip(lo_, hi_).mean() - b.clip(lo_, hi_).mean():.4f}")
infl = (a - a.mean()).abs().sort_values(ascending=False)
print("drop most influential signals:",
      {f"top{k}": round(a.drop(infl.index[:k]).mean() - b.mean(), 4) for k in [1, 3, 5, 10]})

print("\n[C. multiplicity across horizons h0-h5, disp_resid]")
H = [0] + HORIZONS
Y = np.column_stack([alt["disp_resid"].shift(-h).reindex(base.index).values for h in H])
obs = np.array([np.nanmean(Y[SIG & ~np.isnan(Y[:, j]), j]) - np.nanmean(Y[~SIG & ~np.isnan(Y[:, j]), j])
                for j in range(Y.shape[1])])
nrow = len(base)
null = np.empty((N_PERM, len(H)))
for i in range(N_PERM):
    sh = np.roll(SIG, rng.integers(1, nrow))
    for j in range(len(H)):
        y = Y[:, j]
        ok = ~np.isnan(y)
        null[i, j] = np.nanmean(y[sh & ok]) - np.nanmean(y[~sh & ok])
mu, sd = null.mean(0), null.std(0)
zobs = (obs - mu) / sd
znull = np.abs((null - mu) / sd)
print(pd.DataFrame({"h": H, "diff": obs.round(4), "z": zobs.round(2),
                    "p_indiv": [round((znull[:, j] >= abs(zobs[j])).mean(), 4) for j in range(len(H))]
                    }).to_string(index=False))
print(f"\nfamily-wise p (max |z| across 6 horizons): {(znull.max(1) >= np.abs(zobs).max()).mean():.4f}")

print("\n[D. sub-period stability: disp_resid h1]")
rows = []
for name, y0, y1_ in PERIODS:
    w = (base.index.year >= y0) & (base.index.year <= y1_)
    s_, b_ = y_disp[w & SIG].dropna(), y_disp[w & BASE].dropna()
    if len(s_) < 5:
        continue
    ma, mb, t, na, nb = welch(s_, b_)
    rows.append({"period": name, "n_sig": na, "signal": round(ma, 4),
                 "lo_flat": round(mb, 4), "diff": round(ma - mb, 4), "t_welch": round(t, 2)})
print(pd.DataFrame(rows).to_string(index=False))

[A. alternative rotation measures, h1]
   measure  n_sig  n_base  signal    base    diff  t_welch  perm_z  p_perm
disp_resid     79    4323 -0.2474 -0.0672 -0.1803  -3.2907 -4.0768  0.0000
       sd9     92    4843 -5.1567 -4.9996 -0.1570  -2.7597 -3.2673  0.0016
      mad9     92    4843 -5.4257 -5.2842 -0.1415  -2.4909 -2.9311  0.0024
 beta_disp     88    4745 -5.2285 -5.0673 -0.1611  -2.8908 -3.4828  0.0002
  rankcorr     92    4842 -0.0397 -0.0275 -0.0122  -0.2784 -0.3316  0.7432

[B. outlier robustness: disp_resid h1]
mean diff      : -0.1803
median diff    : -0.1278
winsorised 1/99: -0.1790
drop most influential signals: {'top1': np.float64(-0.1981), 'top3': np.float64(-0.2262), 'top5': np.float64(-0.2015), 'top10': np.float64(-0.1696)}

[C. multiplicity across horizons h0-h5, disp_resid]
 h    diff     z  p_indiv
 0 -0.0700 -1.43   0.1484
 1 -0.1964 -4.11   0.0000
 2 -0.0763 -1.56   0.1160
 3 -0.0944 -1.91   0.0544
 4 -0.0457 -0.91   0.3640
 5 -0.1097 -2.27   0.0230

family-wise

In [ ]:
# 11

print("[A. estimation window for z-scores]")
rows = []
for w in [30, 60, 120, 250]:
    dv, ds = np.log(px["VIX"]).diff(), np.log(px["SPX"]).diff()
    zv = (dv / dv.rolling(w, min_periods=int(w * 0.67)).std().shift(1)).reindex(base.index)
    zs_ = (ds / ds.rolling(w, min_periods=int(w * 0.67)).std().shift(1)).reindex(base.index)
    s = ((zv > Z) & (zs_.abs() < F)).values
    b = ((zv <= Z) & (zs_.abs() < F)).values
    mr, br, tr, nr, _ = welch(FS["h1"][s], FS["h1"][b])
    _, _, _, ppr = perm_diff(FS["h1"].values, s)
    md, bd, td, nd, _ = welch(y_disp[s], y_disp[b])
    _, _, pzd, ppd = perm_diff(y_disp.values, s)
    rows.append({"win": w, "n_sig": int(s.sum()), "overlap_60d": int((s & SIG).sum()),
                 "h1_bps": round((mr - br) * 1e4, 1), "p_ret": round(ppr, 3),
                 "n_disp": nd, "disp_diff": round(md - bd, 4),
                 "t_disp": round(td, 2), "perm_z": round(pzd, 2), "p_disp": round(ppd, 4)})
print(pd.DataFrame(rows).to_string(index=False))

print("\n[B. rolling window for the dispersion control regression]")
rows = []
for rw in [500, 750, 1250]:
    res = np.full(len(ctrl), np.nan)
    for i in range(rw, len(ctrl)):
        c, *_ = np.linalg.lstsq(Xc[i - rw:i], yc[i - rw:i], rcond=None)
        res[i] = yc[i] - Xc[i] @ c
    ser = pd.Series(res, index=ctrl.index)
    yy = ser.shift(-1).reindex(base.index)
    md, bd, td, nd, _ = welch(yy[SIG], yy[BASE])
    _, _, pzd, ppd = perm_diff(yy.values, SIG)
    rows.append({"roll_w": rw, "first_valid": str(ser.first_valid_index().date()),
                 "n_sig": nd, "diff": round(md - bd, 4), "t_welch": round(td, 2),
                 "perm_z": round(pzd, 2), "p_perm": round(ppd, 4)})
print(pd.DataFrame(rows).to_string(index=False))

print("\n[C. reversal test restricted to the rotation sample (2002+)]")
w02 = base.index >= rot_raw["disp_resid"].first_valid_index()
rows = []
for ix in INDICES:
    for c in ["h1", "h2", "cum1_5"]:
        s, b = MASKS[ix]["sig"] & w02, MASKS[ix]["base"] & w02
        yy = fwd[ix][c].copy()
        yy[~w02] = np.nan
        ma, mb, t, na, nb = welch(yy[s], yy[b])
        _, _, pz, pp = perm_diff(yy.values, s)
        rows.append({"index": ix, "horizon": c, "n_sig": na,
                     "diff_bps": round((ma - mb) * 1e4, 1), "t_welch": round(t, 2),
                     "perm_z": round(pz, 2), "p_perm": round(pp, 4)})
print(pd.DataFrame(rows).to_string(index=False))

print("\n[D. alternative null: stationary bootstrap vs circular shift]")
yv = y_disp.values
ok = ~np.isnan(yv)
obs = np.nanmean(yv[SIG & ok]) - np.nanmean(yv[~SIG & ok])
nsig = int((SIG & ok).sum())
idx_ok = np.where(ok)[0]
for L in [5, 20, 60]:
    null = np.empty(2000)
    for i in range(2000):
        picked = []
        while len(picked) < nsig:
            st = rng.integers(0, len(idx_ok))
            ln = min(int(rng.geometric(1 / L)), nsig - len(picked))
            picked.extend(idx_ok[(st + np.arange(ln)) % len(idx_ok)])
        m = np.zeros(len(yv), dtype=bool)
        m[np.array(picked[:nsig])] = True
        null[i] = np.nanmean(yv[m & ok]) - np.nanmean(yv[~m & ok])
    mu, sd = null.mean(), null.std()
    print(f"block L={L:3d}  null_sd={sd:.4f}  z={(obs - mu) / sd:6.2f}  "
          f"p={(np.abs(null - mu) >= abs(obs - mu)).mean():.4f}")
print(f"circular shift reference: z=-4.08, p=0.0000  |  obs diff={obs:.4f}")

[A. estimation window for z-scores]
 win  n_sig  overlap_60d  h1_bps  p_ret  n_disp  disp_diff  t_disp  perm_z  p_disp
  30    148          102     2.8  0.832      82    -0.1834   -3.60   -4.10   0.000
  60    137          137     9.3  0.398      79    -0.1803   -3.29   -4.11   0.000
 120    131           91    11.2  0.311      74    -0.1848   -3.53   -4.19   0.000
 250    122           61     3.6  0.808      77    -0.1470   -2.95   -2.98   0.003

[B. rolling window for the dispersion control regression]
 roll_w first_valid  n_sig    diff  t_welch  perm_z  p_perm
    500  2000-12-15     82 -0.1607    -3.18   -3.83  0.0002
    750  2001-12-19     79 -0.1803    -3.29   -4.10  0.0000
   1250  2003-12-15     73 -0.2122    -3.53   -4.27  0.0000

[C. reversal test restricted to the rotation sample (2002+)]
index horizon  n_sig  diff_bps  t_welch  perm_z  p_perm
  SPX      h1     79      22.7     2.35    1.36  0.1656
  SPX      h2     79      17.6     1.50    1.02  0.2888
  SPX  cum1_5     79

In [ ]:
# 12

yv = y_disp.values
ok = ~np.isnan(yv)
obs = np.nanmean(yv[SIG & ok]) - np.nanmean(yv[~SIG & ok])
nsig = int((SIG & ok).sum())
idx_ok = np.where(ok)[0]
N_B = 4000

print("[A. block length sweep: L=1 is plain random sampling]")
rows = []
for L in [1, 2, 3, 5, 10, 20, 40, 60]:
    null = np.empty(N_B)
    for i in range(N_B):
        picked = []
        while len(picked) < nsig:
            st = rng.integers(0, len(idx_ok))
            ln = 1 if L == 1 else min(int(rng.geometric(1 / L)), nsig - len(picked))
            picked.extend(idx_ok[(st + np.arange(ln)) % len(idx_ok)])
        m = np.zeros(len(yv), dtype=bool)
        m[np.array(picked[:nsig])] = True
        null[i] = np.nanmean(yv[m & ok]) - np.nanmean(yv[~m & ok])
    mu, sd = null.mean(), null.std()
    rows.append({"L": L, "null_sd": round(sd, 4), "z": round((obs - mu) / sd, 2),
                 "p": round((np.abs(null - mu) >= abs(obs - mu)).mean(), 4)})
print(pd.DataFrame(rows).to_string(index=False))

print("\n[B. what block length does the signal itself imply?]")
gaps = np.diff(np.where(SIG)[0])
runs = pd.Series(SIG.astype(int))
runlen = runs.groupby((runs != runs.shift()).cumsum()).sum()
runlen = runlen[runlen > 0]
print("mean gap:", round(gaps.mean(), 1), "| mean run length:", round(runlen.mean(), 2),
      "| implied L (mean run):", round(runlen.mean(), 2))
print("run length distribution:", runlen.value_counts().sort_index().to_dict())

print("\n[C. does the block null reproduce the signal's own structure?]")
rows = []
for L in [1, 5, 20, 60]:
    lens = []
    for _ in range(500):
        picked = []
        while len(picked) < nsig:
            st = rng.integers(0, len(idx_ok))
            ln = 1 if L == 1 else min(int(rng.geometric(1 / L)), nsig - len(picked))
            picked.extend(idx_ok[(st + np.arange(ln)) % len(idx_ok)])
        m = np.zeros(len(yv), dtype=bool)
        m[np.array(picked[:nsig])] = True
        r = pd.Series(m.astype(int))
        rl = r.groupby((r != r.shift()).cumsum()).sum()
        lens.append(rl[rl > 0].mean())
    rows.append({"L": L, "mean_run_in_null": round(np.mean(lens), 2),
                 "actual_signal_run": round(runlen.mean(), 2)})
print(pd.DataFrame(rows).to_string(index=False))

print("\n[D. cross-check: two more nulls that preserve different features]")
null_p = np.empty(N_B)
for i in range(N_B):
    m = np.zeros(len(yv), dtype=bool)
    m[rng.choice(idx_ok, nsig, replace=False)] = True
    null_p[i] = np.nanmean(yv[m & ok]) - np.nanmean(yv[~m & ok])
mu, sd = null_p.mean(), null_p.std()
print(f"plain permutation      null_sd={sd:.4f}  z={(obs - mu) / sd:6.2f}  "
      f"p={(np.abs(null_p - mu) >= abs(obs - mu)).mean():.4f}")

yrs = base.index.year.values
null_s = np.empty(N_B)
for i in range(N_B):
    m = np.zeros(len(yv), dtype=bool)
    for yy in np.unique(yrs[SIG]):
        pool = np.where((yrs == yy) & ok)[0]
        k = int(((yrs == yy) & SIG & ok).sum())
        if k and len(pool) >= k:
            m[rng.choice(pool, k, replace=False)] = True
    null_s[i] = np.nanmean(yv[m & ok]) - np.nanmean(yv[~m & ok])
mu, sd = null_s.mean(), null_s.std()
print(f"year-stratified         null_sd={sd:.4f}  z={(obs - mu) / sd:6.2f}  "
      f"p={(np.abs(null_s - mu) >= abs(obs - mu)).mean():.4f}")
print(f"\ncircular shift reference: null_sd=0.0475  z=-4.08  p=0.0000  |  obs={obs:.4f}")

[A. block length sweep: L=1 is plain random sampling]
 L  null_sd     z      p
 1   0.0538 -3.64 0.0002
 2   0.0752 -2.62 0.0100
 3   0.0901 -2.16 0.0305
 5   0.1127 -1.79 0.0710
10   0.1487 -1.32 0.1630
20   0.1928 -1.02 0.2618
40   0.2318 -0.86 0.3272
60   0.2410 -0.80 0.3428

[B. what block length does the signal itself imply?]
mean gap: 67.2 | mean run length: 1.01 | implied L (mean run): 1.01
run length distribution: {1: 135, 2: 1}

[C. does the block null reproduce the signal's own structure?]
 L  mean_run_in_null  actual_signal_run
 1              1.01               1.01
 5              5.00               1.01
20             19.53               1.01
60             43.65               1.01

[D. cross-check: two more nulls that preserve different features]
plain permutation      null_sd=0.0550  z= -3.59  p=0.0008
year-stratified         null_sd=0.0438  z= -3.90  p=0.0003

circular shift reference: null_sd=0.0475  z=-4.08  p=0.0000  |  obs=-0.1964


In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd

SEC = ["XLB","XLE","XLF","XLI","XLK","XLP","XLU","XLV","XLY","XLRE","XLC"]

adj = yf.download(SEC, start="1990-01-01", auto_adjust=True, progress=False, group_by="column")["Close"]
raw = yf.download(SEC, start="1990-01-01", auto_adjust=False, progress=False, group_by="column")

print("[raw columns]", list(raw.columns.get_level_values(0).unique()))

cl = raw["Close"]
ac = raw["Adj Close"] if "Adj Close" in raw.columns.get_level_values(0) else None

print("\n[adjustment factor: auto_adjust Close / unadjusted Close]")
fac = (adj / cl).dropna(how="all")
print(fac.describe().T[["min", "50%", "max"]].round(4).to_string())

if ac is not None:
    print("\n[auto_adjust Close vs Adj Close: max abs relative diff]")
    rel = ((adj - ac).abs() / ac).max()
    print(rel.round(8).to_string())

r_adj = np.log(adj).diff()
r_cl = np.log(cl).diff()
print("\n[return difference: adjusted vs unadjusted, bps]")
dd = (r_adj - r_cl) * 1e4
print(pd.DataFrame({"mean": dd.mean().round(3), "sd": dd.std().round(3),
                    "max_abs": dd.abs().max().round(1),
                    "n_nonzero": (dd.abs() > 1e-8).sum()}).to_string())

print("\n[dispersion: adjusted vs unadjusted]")
d_adj = np.log(r_adj.std(axis=1, ddof=1)).dropna()
d_cl = np.log(r_cl.std(axis=1, ddof=1)).dropna()
common = d_adj.index.intersection(d_cl.index)
print("corr:", round(d_adj[common].corr(d_cl[common]), 6))
print("mean abs diff:", round((d_adj[common] - d_cl[common]).abs().mean(), 6))
print("max abs diff :", round((d_adj[common] - d_cl[common]).abs().max(), 6))

[raw columns] ['Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume']

[adjustment factor: auto_adjust Close / unadjusted Close]
           min     50%  max
Ticker                     
XLB     0.5470  0.7551  1.0
XLC     0.9248  0.9582  1.0
XLE     0.4896  0.6101  1.0
XLF     0.5874  0.7763  1.0
XLI     0.6184  0.7794  1.0
XLK     0.7432  0.8385  1.0
XLP     0.5241  0.6925  1.0
XLRE    0.6905  0.8323  1.0
XLU     0.3886  0.6353  1.0
XLV     0.6678  0.7977  1.0
XLY     0.7351  0.8527  1.0

[auto_adjust Close vs Adj Close: max abs relative diff]
Ticker
XLB     1.310000e-06
XLC     7.200000e-07
XLE     1.250000e-06
XLF     1.230000e-06
XLI     1.290000e-06
XLK     1.270000e-06
XLP     1.440000e-06
XLRE    6.400000e-07
XLU     1.360000e-06
XLV     1.260000e-06
XLY     1.170000e-06

[return difference: adjusted vs unadjusted, bps]
         mean      sd  max_abs  n_nonzero
Ticker                                   
XLB     0.870   7.746    257.2       6838
XLC     0.384   3.103     34.8      